# 1 Import các thư viện cần thiết

In [1]:
import pandas as pd

# 2 Import dữ liệu

In [2]:
def load_sentences(path):
    with open(path, encoding="utf-8") as f:
        return f.read().splitlines()

X_train = load_sentences("UIT_VSFC/train/sents.txt")
X_dev = load_sentences("UIT_VSFC/dev/sents.txt")
X_test = load_sentences("UIT_VSFC/test/sents.txt")

y_train = pd.read_csv("UIT_VSFC/train/sentiments.txt", header=None)[0].values
y_dev = pd.read_csv("UIT_VSFC/dev/sentiments.txt", header=None)[0].values
y_test = pd.read_csv("UIT_VSFC/test/sentiments.txt", header=None)[0].values

# Phase1: Thực hiện các yêu cầu có trong pdf để nắm rõ cơ bản và lý thuyết

## 3. Tokenize tiếng Việt

In [3]:
from pyvi import ViTokenizer

def tokenize(sentences):
    tokenized = []
    for s in sentences:
        tokenized.append(ViTokenizer.tokenize(s))
    return tokenized
X_train_tok = tokenize(X_train)
X_dev_tok = tokenize(X_dev)
X_test_tok = tokenize(X_test)

**Vietnamese Tokenization**

Tiếng Việt là ngôn ngữ đa âm tiết, trong đó một từ có thể gồm nhiều âm tiết được viết cách nhau bởi dấu cách. Vì vậy, nếu chỉ tách văn bản theo khoảng trắng, mô hình sẽ không nhận diện đúng các từ ghép.

Ví dụ:

giảng viên rất nhiệt tình

Nếu tách theo khoảng trắng:

giảng | viên | rất | nhiệt | tình

Tuy nhiên, trong ngữ nghĩa tiếng Việt, các từ đúng phải là:

giảng_viên | rất | nhiệt_tình

Do đó, cần sử dụng bộ tách từ chuyên dụng cho tiếng Việt. Trong bài thực hành này, thư viện **PyVi** được sử dụng để thực hiện tokenization.

## 4. Xây dựng vocabulary

In [4]:
V = []
for t in X_train_tok:
    V += t.split()
V = list(set(V))

print("Vocabulary size:", len(V))

Vocabulary size: 3704


**Building Vocabulary**

Vocabulary là tập hợp tất cả các từ xuất hiện trong dữ liệu huấn luyện.  
Việc xây dựng vocabulary giúp mô hình xác định các từ có thể được xử lý và ánh xạ chúng sang các chỉ số số học.

Quy trình xây dựng vocabulary gồm các bước:

1. Duyệt qua toàn bộ câu trong tập huấn luyện.
2. Tách mỗi câu thành các từ riêng lẻ.
3. Thu thập tất cả các từ vào một danh sách.
4. Loại bỏ các từ trùng lặp.

## 5. Xây dựng dictionary

In [5]:
word_to_index = {w: i+2 for i, w in enumerate(V)}
word_to_index["PAD"] = 0
word_to_index["UNK"] = 1
index_to_word = {i: w for w, i in word_to_index.items()}

**Building Word Dictionary**

Sau khi xây dựng vocabulary, ta cần ánh xạ mỗi từ sang một chỉ số số học.  
Quá trình này được thực hiện thông qua hai từ điển:

- **word_to_index**: ánh xạ từ → chỉ số
- **index_to_word**: ánh xạ chỉ số → từ

# 6. Encode văn bản

In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 100
def encode(sentences):    
    encoded = []    
    for s in sentences:        
        sent = []        
        for w in s.split():            
            w = w.lower()            
            if w in word_to_index:
                sent.append(word_to_index[w])
            else:
                sent.append(word_to_index["UNK"])        
        encoded.append(sent)    
    encoded = pad_sequences(
        encoded,
        maxlen=max_len,
        padding="post",
        value=word_to_index["PAD"]
    )    
    return encoded

**encode dữ liệu**

In [7]:
X_train_enc = encode(X_train_tok)
X_dev_enc = encode(X_dev_tok)
X_test_enc = encode(X_test_tok)

**Encoding Text Data**

Các mô hình học máy không thể xử lý trực tiếp dữ liệu dạng văn bản, vì vậy cần chuyển văn bản sang dạng số.

Quá trình này được gọi là **encoding**.

Mỗi từ trong câu sẽ được thay thế bằng chỉ số tương ứng trong `word_to_index`.

Ví dụ:

"giảng_viên rất nhiệt_tình"

→

[124, 56, 332]

Tuy nhiên, các câu trong dữ liệu có độ dài khác nhau.  
Để đảm bảo dữ liệu có kích thước đồng nhất, ta sử dụng **padding** để thêm các giá trị `PAD` vào cuối câu.


# 7. One-hot label

Ở phần này có 3 nhãn: 0 negative, 1 neutral, 2 positive

In [8]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, num_classes=3)
y_dev_cat = to_categorical(y_dev, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

**One-Hot Encoding Labels**

Các nhãn trong dataset ban đầu được biểu diễn dưới dạng số nguyên.

Ví dụ:

0 → negative  
1 → neutral  
2 → positive  

Để huấn luyện mô hình với hàm mất mát `categorical_crossentropy`, ta cần chuyển nhãn sang dạng **one-hot vector**.

Ví dụ:

0 → [1,0,0]  
1 → [0,1,0]  
2 → [0,0,1]

Quá trình này được thực hiện bằng hàm `to_categorical()` của Keras.

# 8 Xây dựng mô hình

In [9]:
from tensorflow.keras.layers import Dense, Embedding, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

num_words = len(word_to_index)
input_layer = Input(shape=(max_len,))
emb = Embedding(
    input_dim=num_words,
    output_dim=300,
    input_length=max_len
)(input_layer)
flat = Flatten()(emb)
output = Dense(3, activation="softmax")(flat)
model = Model(input_layer, output)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

c:\Users\HOANG\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 300)       │     1,111,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 30000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │        90,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,201,803 (4.58 MB)

 Trainable params: 1,201,803 (4.58 MB)

 Non-trainable params: 0 (0.00 B)

**Building the Neural Network Model**

Mô hình sử dụng kiến trúc đơn giản gồm các lớp:

Input  
↓  
Embedding  
↓  
Flatten  
↓  
Dense (classification layer)

**Input Layer**

Lớp đầu vào nhận chuỗi các chỉ số từ đã được encode.

**Embedding Layer**

Lớp embedding chuyển mỗi từ từ dạng index sang vector dense có kích thước cố định (ở đây là 300 chiều).

Embedding giúp mô hình học được mối quan hệ ngữ nghĩa giữa các từ.

**Flatten Layer**

Lớp Flatten chuyển ma trận embedding thành một vector 1 chiều để đưa vào lớp phân loại.

**Dense Layer**

Lớp Dense cuối cùng sử dụng hàm kích hoạt **softmax** để dự đoán xác suất của từng lớp.

# 9. Train model

In [11]:
history = model.fit(
    X_train_enc,
    y_train_cat,
    validation_data=(X_dev_enc, y_dev_cat),
    batch_size=128,
    epochs=20
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9924 - loss: 0.0383 - val_accuracy: 0.9040 - val_loss: 0.3367
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9930 - loss: 0.0327 - val_accuracy: 0.9046 - val_loss: 0.3451
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.9940 - loss: 0.0286 - val_accuracy: 0.9027 - val_loss: 0.3545
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9954 - loss: 0.0253 - val_accuracy: 0.9033 - val_loss: 0.3672
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9957 - loss: 0.0223 - val_accuracy: 0.9021 - val_loss: 0.3815
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9965 - loss: 0.0199 - val_accuracy: 0.8983 - val_loss: 0.3914
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9962 - loss: 0.0181 - val_accuracy: 0.8983 - val_loss: 0.4057
Epoch 8/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9968 - loss: 0.0168 - val_accuracy: 0.8951 - v

**Model Training**

Sau khi xây dựng mô hình, ta tiến hành huấn luyện bằng phương thức `fit()`.

# 10. Evaluate

In [12]:
loss, acc = model.evaluate(X_test_enc, y_test_cat)
print("Test accuracy:", acc)

99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8797 - loss: 0.7472
Test accuracy: 0.8796588778495789


**Model Evaluation**

Sau khi huấn luyện xong, mô hình được đánh giá trên tập test.

Các chỉ số đánh giá gồm:

* Loss: giá trị hàm mất mát
* Accuracy: độ chính xác của mô hình
* Accuracy cho biết tỷ lệ dự đoán đúng trên toàn bộ dữ liệu kiểm thử.

# 11. Sử dụng pre-trained embedding

Ở phần này lúc đầu em sẽ tải model W2V_ner như trong pdf có nhắc đến bằng cách truy cập vào đường dẫn của github:

https://github.com/vietnlp/etnlp

**Load embedding**

In [13]:
import numpy as np

embeddings_index = {}
embedding_dim = 300
with open("W2V_ner.vec", encoding="utf-8") as f:    
    for line in f:        
        values = line.split()        
        word = values[0]        
        coefs = np.asarray(values[1:], dtype="float32")        
        embeddings_index[word] = coefs

**Pre-trained Word Embedding**

Pre-trained embedding là các vector từ đã được huấn luyện trước trên một tập dữ liệu lớn.

Ưu điểm của việc sử dụng pre-trained embedding:

- cải thiện khả năng biểu diễn ngữ nghĩa của từ
- giúp mô hình học nhanh hơn
- tăng hiệu suất khi dữ liệu huấn luyện nhỏ

Trong bài thực hành này, ta sử dụng bộ **W2V_ner** được huấn luyện sẵn cho tiếng Việt.

# 12. Tạo embedding matrix

In [14]:
num_words = len(word_to_index)
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_to_index.items():    
    vector = embeddings_index.get(word)    
    if vector is not None:
        embedding_matrix[i] = vector

**Building the Embedding Matrix**

Embedding matrix là ma trận chứa vector embedding của toàn bộ từ trong vocabulary.

Kích thước của ma trận:

(num_words, embedding_dim)

Trong đó:

- **num_words**: số từ trong vocabulary
- **embedding_dim**: số chiều của vector embedding

Mỗi hàng trong ma trận tương ứng với vector embedding của một từ.

Nếu từ không tồn tại trong pre-trained embedding, ta có thể gán vector ngẫu nhiên hoặc vector 0.

# 13. Model với pre-trained embedding

In [15]:
from tensorflow.keras.initializers import Constant
input_layer = Input(shape=(max_len,))
emb = Embedding(
    input_dim=num_words,
    output_dim=embedding_dim,
    embeddings_initializer=Constant(embedding_matrix),
    input_length=max_len,
    trainable=True
)(input_layer)
flat = Flatten()(emb)
output = Dense(3, activation="softmax")(flat)
model = Model(input_layer, output)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

**Model with Pre-trained Embedding**

Sau khi tạo embedding matrix, ta đưa ma trận này vào lớp Embedding của mô hình.

# 14. Train lại model

In [16]:
model.fit(
    X_train_enc,
    y_train_cat,
    validation_data=(X_dev_enc, y_dev_cat),
    batch_size=128,
    epochs=10
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.8044 - loss: 0.5080 - val_accuracy: 0.8831 - val_loss: 0.3358
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9110 - loss: 0.2717 - val_accuracy: 0.9040 - val_loss: 0.2831
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9323 - loss: 0.2108 - val_accuracy: 0.9109 - val_loss: 0.2723
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9488 - loss: 0.1686 - val_accuracy: 0.9141 - val_loss: 0.2711
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9599 - loss: 0.1348 - val_accuracy: 0.9109 - val_loss: 0.2769
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9725 - loss: 0.1074 - val_accuracy: 0.9160 - val_loss: 0.2785
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9786 - loss: 0.0871 - val_accuracy: 0.9122 - val_loss: 0.2886
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9826 - loss: 0.0709 - val_accuracy: 0.9103 - v

**Training the Model with Pre-trained Embedding**

Sau khi tích hợp embedding matrix vào mô hình, ta tiến hành huấn luyện lại mô hình tương tự như trước.

# 15. Static vs Jointly

# Phase 2: Bài tập thực hành

Trong phần này, mô hình được áp dụng cho hai nhiệm vụ chính:

1. **Sentiment Classification** – đánh giá cảm xúc của phản hồi sinh viên.
2. **Topic Classification** – phân loại chủ đề của phản hồi.

Hai bài tập giúp kiểm tra khả năng của mô hình trong các bài toán phân loại văn bản khác nhau.

## 1. Bài tập 1 Đánh giá mô hình bằng F1-score

In [17]:
from sklearn.metrics import classification_report
y_pred = model.predict(X_test_enc)
y_pred = y_pred.argmax(axis=1)
print(classification_report(y_test, y_pred))

99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      1409
           1       0.57      0.32      0.41       167
           2       0.91      0.92      0.92      1590

    accuracy                           0.89      3166
   macro avg       0.79      0.72      0.75      3166
weighted avg       0.89      0.89      0.89      3166



**Exercise 1: Evaluating the Model using F1-scorscor**

Sau khi huấn luyện mô hình, cần đánh giá hiệu suất của mô hình trên tập test.

Trong bài toán phân loại văn bản, chỉ số **Accuracy** đôi khi không đủ để phản ánh chất lượng mô hình, đặc biệt khi dữ liệu mất cân bằng giữa các lớp. Vì vậy, chỉ số **F1-score** thường được sử dụng.

F1-score là trung bình điều hòa giữa **Precision** và **Recall**:

F1 = 2 × (Precision × Recall) / (Precision + Recall)

Trong đó:

- **Precision**: tỷ lệ dự đoán đúng trên tổng số dự đoán của lớp đó.
- **Recall**: tỷ lệ dự đoán đúng trên tổng số mẫu thực sự thuộc lớp đó.

Để tính các chỉ số này, ta sử dụng hàm `classification_report` từ thư viện `sklearn`.

## 2. Bài tập 2: Topic Classification

In [18]:
y_train = pd.read_csv("UIT_VSFC/train/topics.txt", header=None)[0].values
y_dev = pd.read_csv("UIT_VSFC/dev/topics.txt", header=None)[0].values
y_test = pd.read_csv("UIT_VSFC/test/topics.txt", header=None)[0].values

In [ ]:
num_classes = len(np.unique(y_train))
input_layer = Input(shape=(max_len,))

emb = Embedding(
    input_dim=num_words,
    output_dim=300,
    input_length=max_len
)(input_layer)

flat = Flatten()(emb)
output = Dense(num_classes, activation="softmax")(flat)
model = Model(input_layer, output)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

c:\Users\HOANG\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
model.fit(
    X_train_enc,
    y_train_cat,
    validation_data=(X_dev_enc, y_dev_cat),
    epochs=10,
    batch_size=128
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.7715 - loss: 0.6022 - val_accuracy: 0.8433 - val_loss: 0.4154
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.8839 - loss: 0.3364 - val_accuracy: 0.8806 - val_loss: 0.3381
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9163 - loss: 0.2516 - val_accuracy: 0.8724 - val_loss: 0.3379
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9386 - loss: 0.1954 - val_accuracy: 0.8749 - val_loss: 0.3440
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.9555 - loss: 0.1517 - val_accuracy: 0.8680 - val_loss: 0.3669
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9665 - loss: 0.1187 - val_accuracy: 0.8598 - val_loss: 0.3835
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9751 - loss: 0.0939 - val_accuracy: 0.8623 - val_loss: 0.4027
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9814 - loss: 0.0759 - val_accuracy: 0.8591 - v

**Exercise 2: Topic Classsification**

Ngoài phân tích cảm xúc, dataset UIT-VSFC còn cung cấp nhãn **chủ đề (topic)** cho mỗi phản hồi của sinh viên.

Bài toán topic classification nhằm xác định chủ đề của phản hồi, ví dụ:

- Cơ sở vật chất
- Giảng viên
- Chương trình học
- Khác

Các nhãn chủ đề được lưu trong file `topics.txt`.

Ngoài ra, ta cần **Loading topic labels**

Để thực hiện bài toán này, ta đọc dữ liệu nhãn chủ đề từ dataset.


# Debug

In [20]:
import numpy as np

print(np.unique(y_train))

[0 1 2 3]
